In [ ]:
import re
import pandas as pd

In [ ]:
import json as _json_mod
import os as _os_mod

def load_config(config_path=None):
    """Load configuration from config.json.
    
    Searches for config.json in the current directory, then parent directories.
    Returns a flat dict with keys like 'dbx_catalog', 'sf_account_locator', etc.
    for backward compatibility with notebook widgets.
    """
    if config_path is None:
        # Search current dir and up to 3 parent dirs
        for d in [_os_mod.getcwd()] + [_os_mod.path.join(_os_mod.getcwd(), ".." * i) for i in range(1, 4)]:
            candidate = _os_mod.path.join(_os_mod.path.abspath(d), "config.json")
            if _os_mod.path.exists(candidate):
                config_path = candidate
                break
        if config_path is None:
            raise FileNotFoundError(
                "config.json not found. Copy config.json.example to config.json and fill in your values."
            )

    with open(config_path) as f:
        raw = _json_mod.load(f)

    # Flatten to backward-compatible keys
    config = {}
    dbx = raw.get("databricks", {})
    config["dbx_catalog"] = dbx.get("catalog", "")
    config["dbx_schema"] = dbx.get("schema", "")
    config["dbx_volume"] = dbx.get("volume", "")
    config["dbx_pat_token"] = dbx.get("pat_token", "")
    config["dbx_hourly_rate"] = dbx.get("hourly_rate", "2.00")
    config["dbx_optimize_strategy"] = dbx.get("optimize_strategy", "zorder")

    sf = raw.get("snowflake", {})
    config["sf_account_locator"] = sf.get("account_locator", "")
    config["sf_region"] = sf.get("region", "")
    config["sf_database"] = sf.get("database", "")
    config["sf_schema"] = sf.get("schema", "")
    config["sf_staging"] = sf.get("staging", "")
    config["sf_external_volume"] = sf.get("external_volume", "")
    config["sf_role"] = sf.get("role", "")
    config["sf_user"] = sf.get("user", "")
    config["sf_auth_key"] = sf.get("auth_key", "")
    config["sf_hourly_rate"] = sf.get("hourly_rate", "4.00")

    ml = raw.get("mlflow", {})
    config["mlflow_experiment_name"] = ml.get("experiment_name", "/TPC-DS Benchmark")

    print(f"Config loaded from {config_path}")
    return config


def get_volume_base(cfg):
    """Return the base path for the unified volume."""
    return f"/Volumes/{cfg['dbx_catalog']}/{cfg['dbx_schema']}/{cfg['dbx_volume']}"

In [ ]:
import os
import re
import subprocess

def load_vars_from_bash(file_path=None):
    """
    Parses a Bash 'declare -p' output file, sets the variables
    as environment variables, and then deletes the file.

    Args:
        file_path (str, optional): The absolute path to the state file.
                                   Defaults to '~/.cell_vars'.
    """
    if file_path is None:
        # Use os.path.expanduser to correctly handle the '~' home directory symbol
        file_path = os.path.join("/tmp/", f"{execution_id}.state")

    print(f"ⓘ Attempting to load variables from: {file_path}")

    # A regular expression to capture VAR="VALUE" from a 'declare' line.
    # It handles declarations like 'declare -x VAR="value"' or 'declare -- VAR="value"'.
    # It captures the variable name in group 1 and the value (inside quotes) in group 2.
    # This regex specifically looks for simple string assignments.
    declare_pattern = re.compile(r'export (\w+)="([^"]*)"')

    try:
        # Ensure the file exists before trying to open it
        if not os.path.exists(file_path):
            print(f"ℹ No state file found at: {file_path}. No variables loaded.")
            return

        with open(file_path, 'r') as f:
            lines = f.readlines()

        loaded_count = 0
        for line in lines:
            match = declare_pattern.match(line)
            if match:
                var_name = match.group(1)
                var_value = match.group(2)
                # Set the environment variable for the current process
                os.environ[var_name] = var_value
                loaded_count += 1
        
        if loaded_count > 0:
            print(f"✔ Successfully loaded {loaded_count} variable(s) into the environment.")
        else:
            print("ℹ State file was empty or contained no simple string variables to load.")

        try:
            os.remove(file_path)
            print(f"✔ Source file '{file_path}' has been deleted.")
        except OSError as e:
            print(f"❌ Error deleting file '{file_path}': {e}")

    except Exception as e:
        print(f"❌ An error occurred while reading the file: {e}")


In [ ]:
import os

pid = os.getpid()
notebook_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookId().get()
execution_id = f"{pid}-{notebook_id}"
os.environ["EXECUTION_ID"] = execution_id

In [ ]:
import csv as csv_mod

def persist_to_tsv(queries: dict, file_path: str):
    """Persist a {query_name: query_text} dictionary to a TSV file."""
    with open(file_path, 'w', newline='') as f:
        writer = csv_mod.writer(f, delimiter='\t', quoting=csv_mod.QUOTE_NONE, escapechar=None, quotechar=None)
        writer.writerow(['query_name', 'query'])
        for name, sql in queries.items():
            writer.writerow([name, sql])

In [ ]:
import re as _re_mod

TPCDS_TABLE_NAMES = [
    "call_center", "catalog_page", "catalog_returns", "catalog_sales",
    "customer_address", "customer_demographics", "customer",
    "date_dim", "household_demographics", "income_band", "inventory",
    "item", "promotion", "reason", "ship_mode",
    "store_returns", "store_sales", "store",
    "time_dim", "warehouse", "web_page", "web_returns", "web_sales", "web_site",
]

# Pre-sorted by length descending so 'store_returns' is matched before 'store'
_TPCDS_TABLES_SORTED = sorted(TPCDS_TABLE_NAMES, key=len, reverse=True)

def prefix_table_names(sql, prefix):
    """Replace all TPC-DS table names in SQL with prefixed versions.

    Uses word-boundary regex to avoid corrupting column names (cs_item_sk),
    string literals ('store'), or other identifiers. Longer table names are
    replaced first to prevent partial matches.
    """
    if not prefix:
        return sql
    for table in _TPCDS_TABLES_SORTED:
        pattern = r"(?<![a-zA-Z0-9_\'])" + _re_mod.escape(table) + r"(?![a-zA-Z0-9_\'])"
        sql = _re_mod.sub(pattern, prefix + table, sql)
    return sql

In [ ]:
import csv, json, os, shutil

def generate_concurrency_streams(perf_queries_tsv, concurrency_json, output_dir):
    """Generate per-stream TSV files for JMeter concurrency testing.

    Reads the performance query TSV and concurrency query-order JSON,
    then writes one TSV per stream (stream_1.tsv .. stream_20.tsv) into
    output_dir. Each TSV has columns: query_id, query_text, stream_id.

    TPC-DS queries with a/b variants (e.g. q14a, q14b) are expanded so
    both parts run when that query number appears in the stream.
    """
    # Build query_name -> query_text lookup from performance queries
    queries = {}
    with open(perf_queries_tsv, 'r') as f:
        reader = csv.reader(f, delimiter='\t', quoting=csv.QUOTE_NONE)
        next(reader)  # skip header
        for row in reader:
            queries[row[0]] = row[1]

    # Read stream orderings
    with open(concurrency_json, 'r') as f:
        streams = json.load(f)['streams']

    # Delete and recreate output directory to avoid stale files
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    generated = 0
    for stream in streams:
        stream_id = stream['stream_id']
        if stream_id < 1 or stream_id > 20:
            continue

        filepath = os.path.join(output_dir, f"stream_{stream_id}.tsv")
        with open(filepath, 'w', newline='') as f:
            writer = csv.writer(f, delimiter='\t', quoting=csv.QUOTE_NONE, escapechar=None, quotechar=None)
            writer.writerow(['query_id', 'query_text', 'stream_id'])
            for query_num in stream['query_order']:
                query_name = f"q{query_num}"
                if query_name in queries:
                    writer.writerow([query_name, queries[query_name], stream_id])
                else:
                    # Queries with a/b variants (e.g. q14a, q14b)
                    found = False
                    for suffix in ('a', 'b'):
                        variant = f"{query_name}{suffix}"
                        if variant in queries:
                            writer.writerow([variant, queries[variant], stream_id])
                            found = True
                    if not found:
                        raise ValueError(f"Query '{query_name}' (and a/b variants) from stream {stream_id} not found in {perf_queries_tsv}")
        generated += 1

    print(f"Generated {generated} stream files in {output_dir}")

### MLflow Tracking Utilities

In [ ]:
import hashlib
import json as json_mod
import os
import re as re_mod
import shutil
import subprocess
from datetime import datetime, timezone




def strategy_slug(raw):
    """Slugify a strategy value for use in SQL identifiers and warehouse names.
    Examples:
        'liquid clustering'   -> 'LIQUID_CLUSTERING'
        'iceberg_zordered'    -> 'ICEBERG_ZORDERED'
        'native_gen2'         -> 'NATIVE_GEN2'
    Always upper-case, non-empty (returns 'DEFAULT' for None/empty input).
    """
    return (re.sub(r"\W+", "_", (raw or "").strip()).strip("_") or "DEFAULT").upper()


def compute_md5(file_path):
    """Compute MD5 hex digest of a file."""
    h = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def _compute_duration_from_csv(csv_path):
    """Compute wall-clock duration in seconds from a JMeter result CSV."""
    import csv as csv_mod
    first_ts = None
    last_ts = None
    last_elapsed = 0
    with open(csv_path, "r") as f:
        reader = csv_mod.DictReader(f)
        for row in reader:
            ts = int(row["timeStamp"])
            elapsed = int(row["elapsed"])
            if first_ts is None or ts < first_ts:
                first_ts = ts
            if last_ts is None or ts > last_ts:
                last_ts = ts
                last_elapsed = elapsed
    if first_ts is not None and last_ts is not None:
        return (last_ts - first_ts + last_elapsed) / 1000.0
    return 0.0


def build_test_config(platform, test_type, jmx_file, tsv_files, jdbc_driver):
    """Build a test configuration dict with MD5 checksums and start timestamp."""
    return {
        "test_start_time": datetime.now(timezone.utc).isoformat(),
        "platform": platform,
        "test_type": test_type,
        "checksums": {
            "jmx": {
                "file": os.path.basename(jmx_file),
                "md5": compute_md5(jmx_file),
            },
            "tsv": [
                {"file": os.path.basename(f), "md5": compute_md5(f)}
                for f in tsv_files
            ],
            "jdbc_driver": {
                "file": os.path.basename(jdbc_driver),
                "md5": compute_md5(jdbc_driver),
            },
        },
    }


def get_databricks_warehouse_config(warehouse_url, pat_token):
    """Fetch warehouse size, type, and channel from the Databricks REST API."""
    from urllib.parse import urlparse
    import requests

    parsed = urlparse(warehouse_url.replace("jdbc:databricks:", "https:"))
    host = parsed.hostname
    warehouse_id = parsed.path.rstrip("/").split("/")[-1]

    try:
        resp = requests.get(
            f"https://{host}/api/2.0/sql/warehouses/{warehouse_id}",
            headers={"Authorization": f"Bearer {pat_token}"},
            timeout=30,
        )
        resp.raise_for_status()
        wh = resp.json()
        return {
            "warehouse_name": wh.get("name", ""),
            "warehouse_size": wh.get("cluster_size", ""),
            "warehouse_type": (
                "serverless"
                if wh.get("enable_serverless_compute")
                else wh.get("warehouse_type", "PRO")
            ),
            "channel": wh.get("channel", {}).get("name", "CHANNEL_NAME_CURRENT"),
            "num_clusters": str(wh.get("num_clusters", 1)),
        }
    except Exception as e:
        print(f"Warning: Could not fetch Databricks warehouse config: {e}")
        return {
            "warehouse_name": "",
            "warehouse_size": "",
            "warehouse_type": "",
            "channel": "",
            "num_clusters": "",
        }


def get_snowflake_warehouse_config(sf_account_locator, sf_region, sf_warehouse,
                                    sf_user, sf_auth_key_path, sf_role,
                                    sf_database, sf_schema):
    """Fetch warehouse size and type from Snowflake via SHOW WAREHOUSES."""
    try:
        import snowflake.connector
        from cryptography.hazmat.primitives import serialization
        from cryptography.hazmat.backends import default_backend

        with open(sf_auth_key_path, "rb") as f:
            p_key = serialization.load_pem_private_key(
                f.read(), password=None, backend=default_backend(),
            )
        pkb = p_key.private_bytes(
            encoding=serialization.Encoding.DER,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption(),
        )
        conn = snowflake.connector.connect(
            user=sf_user,
            account=sf_account_locator if "-" in sf_account_locator else f"{sf_account_locator}.{sf_region}",
            private_key=pkb,
            role=sf_role,
            database=sf_database,
            schema=sf_schema,
            warehouse=sf_warehouse,
        )
        try:
            cur = conn.cursor()
            cur.execute(f"SHOW WAREHOUSES LIKE '{sf_warehouse}'")
            cols = [d[0].lower() for d in cur.description]
            rows = cur.fetchall()
            if rows:
                wh = dict(zip(cols, rows[0]))
                return {
                    "warehouse_name": wh.get("name", sf_warehouse),
                    "warehouse_size": wh.get("size", ""),
                    "warehouse_type": wh.get("type", "STANDARD"),
                    "auto_suspend": str(wh.get("auto_suspend", "")),
                    "auto_resume": str(wh.get("auto_resume", "")),
                }
        finally:
            conn.close()
    except Exception as e:
        print(f"Warning: Could not fetch Snowflake warehouse config: {e}")

    return {
        "warehouse_name": sf_warehouse,
        "warehouse_size": "",
        "warehouse_type": "",
    }


def run_jmeter_with_tracking(
    platform,
    test_type,
    jmx_file,
    tsv_files,
    jdbc_driver,
    result_csvs,
    log_file,
    run_params,
    experiment_name,
    workspace_results_dir,
    workspace_logs_dir,
    result_prefix="",
    run_id="",
):
    """Run a JMeter test with full MLflow tracking.

    Creates an MLflow run that logs parameters, artifacts (test_config.json,
    sanitised JMX, TSV files, result CSVs, JMeter log), and a duration_seconds
    metric computed from the result CSV timestamps.
    """
    import mlflow

    def _add_run_id_suffix(name):
        """Insert __{run_id} before the file extension; no-op if run_id empty."""
        if not run_id:
            return name
        base, ext = os.path.splitext(name)
        # Result CSVs are named "...-report.csv": insert before the "-report" tag
        if base.endswith("-report"):
            return f"{base[:-len('-report')]}__{run_id}-report{ext}"
        return f"{base}__{run_id}{ext}"

    tpcds_dir = "/tmp/tpcds"
    mlflow.set_experiment(experiment_name)

    run_name = f"{result_prefix}{platform}_{test_type}" if result_prefix else f"{platform}_{test_type}"

    with mlflow.start_run(run_name=run_name) as run:
        # --- Log parameters ---
        mlflow.log_param("platform", platform)
        mlflow.log_param("test_type", test_type)
        for k, v in run_params.items():
            mlflow.log_param(k, v)

        # --- Build and log test_config.json ---
        test_config = build_test_config(
            platform, test_type, jmx_file, tsv_files, jdbc_driver,
        )
        config_path = os.path.join(tpcds_dir, "test_config.json")
        with open(config_path, "w") as f:
            json_mod.dump(test_config, f, indent=2)
        mlflow.log_artifact(config_path)

        # --- Log sanitised JMX (redact JDBC passwords) ---
        with open(jmx_file) as f:
            jmx_content = f.read()
        sanitized = re_mod.sub(
            r'(<stringProp name="password">)[^<]*(</stringProp>)',
            r'\1***REDACTED***\2',
            jmx_content,
        )
        sanitized_path = jmx_file + ".sanitized.jmx"
        with open(sanitized_path, "w") as f:
            f.write(sanitized)
        mlflow.log_artifact(sanitized_path, "inputs")
        os.remove(sanitized_path)

        # --- Log input TSV files ---
        for tsv in tsv_files:
            if os.path.exists(tsv):
                mlflow.log_artifact(tsv, "inputs")

        # --- Clean stale result files (JMeter appends to existing CSVs) ---
        for csv_path in result_csvs:
            if os.path.exists(csv_path):
                os.remove(csv_path)
        if os.path.exists(log_file):
            os.remove(log_file)

        # --- Run JMeter ---
        cmd = [
            "jmeter", "-n",
            "-t", jmx_file,
            "-Jxstream.allowed_types=org.apache.jmeter.save.ScriptWrapper",
            "-j", log_file,
        ]
        print(f"Starting JMeter: {platform} / {test_type}")
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=tpcds_dir,
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait()

        # --- Compute duration and log results ---
        total_duration = 0.0
        for csv_path in result_csvs:
            if os.path.exists(csv_path):
                total_duration += _compute_duration_from_csv(csv_path)
                mlflow.log_artifact(csv_path, "results")

        mlflow.log_metric("duration_seconds", total_duration)

        if os.path.exists(log_file):
            mlflow.log_artifact(log_file, "logs")

        # --- Copy results back to workspace ---
        os.makedirs(workspace_results_dir, exist_ok=True)
        os.makedirs(workspace_logs_dir, exist_ok=True)
        for csv_path in result_csvs:
            if os.path.exists(csv_path):
                base = os.path.basename(csv_path)
                dest_name = (result_prefix + base) if result_prefix else base
                dest_name = _add_run_id_suffix(dest_name)
                shutil.copy2(csv_path, os.path.join(workspace_results_dir, dest_name))
        if os.path.exists(log_file):
            base = os.path.basename(log_file)
            dest_name = (result_prefix + base) if result_prefix else base
            dest_name = _add_run_id_suffix(dest_name)
            shutil.copy2(log_file, os.path.join(workspace_logs_dir, dest_name))

        if proc.returncode != 0:
            mlflow.set_tag("jmeter_status", "FAILED")
            raise RuntimeError(
                f"JMeter exited with code {proc.returncode}. "
                f"Check log: {log_file}"
            )
        mlflow.set_tag("jmeter_status", "SUCCESS")

        print(f"\nMLflow run: {run.info.run_id}")
        print(f"Duration: {total_duration:.1f}s")

    return run.info.run_id

def find_or_create_databricks_warehouse(warehouse_size, scale, strategy_slug_value, host, pat_token, max_clusters=1):
    """Find or create a Databricks SQL warehouse named
    TPCDS_{scale}_{size}_{strategy_slug}. Uses the PREVIEW channel.

    `strategy_slug_value` must already be slugified (uppercase, no spaces) —
    typically produced by `strategy_slug()` and passed through widgets.

    Distinct (scale, size, strategy) combos always get distinct warehouses so
    parallel benchmark runs never reuse compute.
    Returns: (warehouse_id, jdbc_url, warehouse_config_dict)
    """
    import requests

    size_map = {
        "2XS": "2X-Small", "XS": "X-Small", "S": "Small", "M": "Medium",
        "L": "Large", "XL": "X-Large", "2XL": "2X-Large",
        "3XL": "3X-Large", "4XL": "4X-Large",
    }
    cluster_size = size_map.get(warehouse_size.upper(), warehouse_size)
    slug = (strategy_slug_value or "DEFAULT").strip().upper() or "DEFAULT"
    wh_name = f"TPCDS_{scale}_{warehouse_size.upper()}_{slug}"
    headers = {"Authorization": f"Bearer {pat_token}"}

    base_cfg = {
        "name": wh_name,
        "cluster_size": cluster_size,
        # 1 minute = the serverless SQL minimum, matching Snowflake's AUTO_SUSPEND=60s
        # floor so idle/auto-stop overhead is comparable across platforms.
        "auto_stop_mins": 1,
        "min_num_clusters": 1,
        "max_num_clusters": max_clusters,
        "warehouse_type": "PRO",
        "enable_serverless_compute": True,
        "channel": {"name": "CHANNEL_NAME_PREVIEW"},
    }

    # Exact-name lookup
    resp = requests.get(
        f"https://{host}/api/2.0/sql/warehouses",
        headers=headers, timeout=30,
    )
    resp.raise_for_status()
    for wh in resp.json().get("warehouses", []):
        if wh["name"] == wh_name:
            wh_id = wh["id"]
            if (wh.get("cluster_size") != cluster_size
                or wh.get("channel", {}).get("name") != "CHANNEL_NAME_PREVIEW"):
                print(f"Reconciling warehouse {wh_name} -> {cluster_size}, PREVIEW channel")
                resp2 = requests.post(
                    f"https://{host}/api/2.0/sql/warehouses/{wh_id}/edit",
                    headers=headers, timeout=30, json=base_cfg,
                )
                resp2.raise_for_status()
            jdbc_url = f"jdbc:databricks://{host}:443/sql/1.0/warehouses/{wh_id}"
            config = get_databricks_warehouse_config(jdbc_url, pat_token)
            print(f"Using existing warehouse: {wh_name} (id={wh_id}, size={cluster_size})")
            return wh_id, jdbc_url, config

    print(f"Creating warehouse: {wh_name} ({cluster_size}, PREVIEW)")
    resp = requests.post(
        f"https://{host}/api/2.0/sql/warehouses",
        headers=headers, timeout=30, json=base_cfg,
    )
    resp.raise_for_status()
    wh_id = resp.json()["id"]
    jdbc_url = f"jdbc:databricks://{host}:443/sql/1.0/warehouses/{wh_id}"
    config = get_databricks_warehouse_config(jdbc_url, pat_token)
    print(f"Created warehouse: {wh_name} (id={wh_id})")
    return wh_id, jdbc_url, config


def find_or_create_snowflake_warehouse(warehouse_size, scale, strategy_slug_value, warehouse_gen,
                                        sf_account_locator, sf_region,
                                        sf_user, sf_auth_key_path, sf_role,
                                        sf_database, sf_schema, max_clusters=1):
    """Find or create a Snowflake warehouse named
    TPCDS_{scale}_{size}_{strategy_slug}. Explicitly sets
    GENERATION = '1' or '2' based on
    `warehouse_gen` (1 or 2).

    `strategy_slug_value` must already be slugified (e.g. NATIVE_GEN1).
    Returns: (warehouse_name, warehouse_config_dict)
    """
    sf_size_map = {
        "2XS": "XSMALL", "XS": "XSMALL", "S": "SMALL", "M": "MEDIUM",
        "L": "LARGE", "XL": "XLARGE", "2XL": "XXLARGE",
        "3XL": "XXXLARGE", "4XL": "X4LARGE",
    }
    sf_size = sf_size_map.get(warehouse_size.upper(), warehouse_size.upper())
    slug = (strategy_slug_value or "DEFAULT").strip().upper() or "DEFAULT"
    wh_name = f"TPCDS_{scale}_{warehouse_size.upper()}_{slug}"

    generation = "2" if int(warehouse_gen) == 2 else "1"

    import snowflake.connector
    from cryptography.hazmat.primitives import serialization
    from cryptography.hazmat.backends import default_backend

    with open(sf_auth_key_path, "rb") as f:
        p_key = serialization.load_pem_private_key(f.read(), password=None, backend=default_backend())
    pkb = p_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    )
    account = sf_account_locator if "-" in sf_account_locator else f"{sf_account_locator}.{sf_region}"
    conn = snowflake.connector.connect(
        user=sf_user, account=account, private_key=pkb,
        role=sf_role, database=sf_database, schema=sf_schema,
    )
    try:
        cur = conn.cursor()
        cur.execute(f"SHOW WAREHOUSES LIKE '{wh_name}'")
        exists = bool(cur.fetchall())
        _is_adaptive = (strategy_slug_value or "").upper().endswith("_ADAPTIVE")
        if not exists:
            if _is_adaptive:
                print(f"Creating Snowflake ADAPTIVE warehouse: {wh_name} (cap={sf_size}, throughput={max_clusters})")
                # === Adaptive warehouse: no WAREHOUSE_SIZE / GENERATION; system auto-scales ===
                # MAX_QUERY_PERFORMANCE_LEVEL = size cap; QUERY_THROUGHPUT_MULTIPLIER = scale factor (0=unlimited, >=2).
                cur.execute(f"""
                    CREATE WAREHOUSE {wh_name}
                    WAREHOUSE_TYPE = 'ADAPTIVE'
                    MAX_QUERY_PERFORMANCE_LEVEL = {sf_size}
                    QUERY_THROUGHPUT_MULTIPLIER = {max_clusters}
                """)
            else:
                print(f"Creating Snowflake warehouse: {wh_name} ({sf_size}, GEN{generation})")
                cur.execute(f"""
                    CREATE WAREHOUSE {wh_name}
                    WAREHOUSE_SIZE = '{sf_size}'
                    GENERATION = '{generation}'
                    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE
                    MAX_CLUSTER_COUNT = {max_clusters} MIN_CLUSTER_COUNT = 1
                """)
        else:
            if _is_adaptive:
                # Reconcile adaptive params (warehouse is already ADAPTIVE; no SIZE/GENERATION)
                cur.execute(f"ALTER WAREHOUSE {wh_name} SET MAX_QUERY_PERFORMANCE_LEVEL = {sf_size}")
                cur.execute(f"ALTER WAREHOUSE {wh_name} SET QUERY_THROUGHPUT_MULTIPLIER = {max_clusters}")
                print(f"Using existing Snowflake ADAPTIVE warehouse: {wh_name}")
            else:
                # Reconcile size + generation
                cur.execute(f"ALTER WAREHOUSE {wh_name} SET WAREHOUSE_SIZE = '{sf_size}'")
                cur.execute(f"ALTER WAREHOUSE {wh_name} SET GENERATION = '{generation}'")
                cur.execute(f"ALTER WAREHOUSE {wh_name} SET MAX_CLUSTER_COUNT = {max_clusters} MIN_CLUSTER_COUNT = 1")
                print(f"Using existing Snowflake warehouse: {wh_name} ({sf_size}, GEN{generation})")
    finally:
        conn.close()

    config = get_snowflake_warehouse_config(
        sf_account_locator, sf_region, wh_name, sf_user,
        sf_auth_key_path, sf_role, sf_database, sf_schema,
    )
    return wh_name, config

### JMX Templates & Platform Configuration

In [ ]:
import re

_PLATFORM_CONFIGS = {
    "DATABRICKS": {
        "PLATFORM": "databricks",
        "DATASOURCE_NAME": "DatabricksSQL",
        "CONNECTION_LABEL": "Connect to Databricks SQL",
        "INIT_QUERY": "SET use_cached_result = false;",
        "JDBC_URL": "{DBX_JDBC_URL}",
        "JDBC_DRIVER_CLASS": "com.databricks.client.jdbc.Driver",
        "JDBC_AUTH_BLOCK": '        <stringProp name="username">token</stringProp>\n        <stringProp name="password">{DBX_WAREHOUSE_TOKEN}</stringProp>\n        <stringProp name="connectionProperties"></stringProp>',
    },
    "SNOWFLAKE": {
        "PLATFORM": "snowflake",
        "DATASOURCE_NAME": "Snowflake",
        "CONNECTION_LABEL": "Connect to Snowflake",
        "INIT_QUERY": "ALTER SESSION SET USE_CACHED_RESULT = FALSE;",
        "JDBC_URL": "jdbc:snowflake://{SF_JDBC_HOST}/?warehouse={SF_WAREHOUSE}&amp;db={SF_DATABASE}&amp;schema={SF_SCHEMA}&amp;role={SF_ROLE}&amp;private_key_file={SF_AUTH_KEY}",
        "JDBC_DRIVER_CLASS": "net.snowflake.client.jdbc.SnowflakeDriver",
        "JDBC_AUTH_BLOCK": '        <stringProp name="username">{SF_USER}</stringProp>\n        <stringProp name="password"></stringProp>\n        <stringProp name="connectionProperties">private_key_file={SF_AUTH_KEY}</stringProp>',
    },
}

def get_jmx_template(platform, test_type):
    """Return a JMX template with platform-specific values injected.

    Args:
        platform: "DATABRICKS" or "SNOWFLAKE"
        test_type: "INGESTION", "POWER_TEST", or "CONCURRENCY"

    Returns:
        A template string with platform parts resolved and user-specific
        placeholders ({CURPATH}, {JDBC_DRIVER}, credentials) still present.
    """
    templates = {"INGESTION": _INGESTION_TEMPLATE, "POWER_TEST": _POWER_TEST_TEMPLATE, "CONCURRENCY": _CONCURRENCY_TEMPLATE}
    if test_type not in templates:
        raise ValueError(f"Unknown test_type '{test_type}'. Expected: {list(templates)}")
    if platform not in _PLATFORM_CONFIGS:
        raise ValueError(f"Unknown platform '{platform}'. Expected: {list(_PLATFORM_CONFIGS)}")

    template = templates[test_type]
    for key, value in _PLATFORM_CONFIGS[platform].items():
        template = template.replace("{" + key + "}", value)
    return template

def fill_template(template, params):
    """Validate params against template placeholders, then interpolate.

    Extracts {placeholder} names from the template (skipping escaped {{...}})
    and checks that every placeholder is provided and non-None in params, and
    that params contains no extra keys. Raises ValueError on mismatch.
    """
    required = set(re.findall(r'(?<!\{)\{(\w+)\}(?!\})', template))
    provided = set(params.keys())

    missing = required - provided
    extra = provided - required
    null_vals = {k for k, v in params.items() if v is None and k in required}

    errors = []
    if missing:
        errors.append(f"Missing template parameters: {sorted(missing)}")
    if null_vals:
        errors.append(f"Parameters with None value: {sorted(null_vals)}")
    if extra:
        errors.append(f"Extra parameters not in template: {sorted(extra)}")
    if errors:
        raise ValueError("\n".join(errors))

    return template.format_map(params)

In [ ]:
_INGESTION_TEMPLATE = """
<?xml version="1.0" encoding="UTF-8"?>
<jmeterTestPlan version="1.2" properties="5.0" jmeter="5.6.3">
  <hashTree>
    <TestPlan guiclass="TestPlanGui" testclass="TestPlan" testname="TPC-DS Ingestion Test Plan" enabled="true">
      <stringProp name="TestPlan.comments"></stringProp>
      <boolProp name="TestPlan.functional_mode">false</boolProp>
      <boolProp name="TestPlan.tearDown_on_shutdown">true</boolProp>
      <boolProp name="TestPlan.serialize_threadgroups">true</boolProp>
      <elementProp name="TestPlan.user_defined_variables" elementType="Arguments" guiclass="ArgumentsPanel" testclass="Arguments" testname="User Defined Variables" enabled="true">
        <collectionProp name="Arguments.arguments"/>
      </elementProp>
      <stringProp name="TestPlan.user_define_classpath">{JDBC_DRIVER}</stringProp>
    </TestPlan>
    <hashTree>
      <JDBCDataSource guiclass="TestBeanGUI" testclass="JDBCDataSource" testname="{CONNECTION_LABEL}" enabled="true">
        <stringProp name="dataSource">{DATASOURCE_NAME}</stringProp>
        <stringProp name="poolMax">1</stringProp>
        <stringProp name="timeout">600000</stringProp>
        <stringProp name="trimInterval">60000</stringProp>
        <boolProp name="autocommit">true</boolProp>
        <stringProp name="transactionIsolation">DEFAULT</stringProp>
        <boolProp name="preinit">true</boolProp>
        <stringProp name="initQuery">{INIT_QUERY}</stringProp>
        <boolProp name="keepAlive">true</boolProp>
        <stringProp name="connectionAge">50000000</stringProp>
        <stringProp name="checkQuery">select 1</stringProp>
        <stringProp name="dbUrl">{JDBC_URL}</stringProp>
        <stringProp name="driver">{JDBC_DRIVER_CLASS}</stringProp>
{JDBC_AUTH_BLOCK}
        <stringProp name="TestPlan.comments"></stringProp>
      </JDBCDataSource>
      <hashTree/>
      <!-- ==================================================================================== -->
      <!-- ==                     PHASE 1: CREATE TABLES (Serial)                            == -->
      <!-- ==================================================================================== -->
      <ThreadGroup guiclass="ThreadGroupGui" testclass="ThreadGroup" testname="Phase 1 - Create Tables" enabled="true">
        <stringProp name="ThreadGroup.on_sample_error">continue</stringProp>
        <elementProp name="ThreadGroup.main_controller" elementType="LoopController" guiclass="LoopControlPanel" testclass="LoopController" testname="Loop Controller" enabled="true">
          <stringProp name="LoopController.loops">-1</stringProp>
          <boolProp name="LoopController.continue_forever">true</boolProp>
        </elementProp>
        <stringProp name="ThreadGroup.num_threads">1</stringProp>
        <stringProp name="ThreadGroup.ramp_time">1</stringProp>
        <boolProp name="ThreadGroup.delayedStart">false</boolProp>
        <boolProp name="ThreadGroup.scheduler">false</boolProp>
        <stringProp name="ThreadGroup.duration"></stringProp>
        <stringProp name="ThreadGroup.delay"></stringProp>
        <boolProp name="ThreadGroup.same_user_on_next_iteration">false</boolProp>
      </ThreadGroup>
      <hashTree>
        <CSVDataSet guiclass="TestBeanGUI" testclass="CSVDataSet" testname="CSV Data Set Config - Create Tables" enabled="true">
          <stringProp name="delimiter">\t</stringProp>
          <stringProp name="fileEncoding">UTF-8</stringProp>
          <stringProp name="filename">{CURPATH}/test_conf/tpcds-queries-ingestion-{PLATFORM}.tsv</stringProp>
          <boolProp name="ignoreFirstLine">true</boolProp>
          <boolProp name="quotedData">false</boolProp>
          <boolProp name="recycle">false</boolProp>
          <stringProp name="shareMode">shareMode.thread</stringProp>
          <boolProp name="stopThread">true</boolProp>
          <stringProp name="variableNames">query_name,query</stringProp>
        </CSVDataSet>
        <hashTree/>
          <JDBCSampler guiclass="TestBeanGUI" testclass="JDBCSampler" testname="JDBC Request - Create Table" enabled="true">
            <stringProp name="dataSource">{DATASOURCE_NAME}</stringProp>
            <stringProp name="query">/* ${{query_name}} */
${{query}}</stringProp>
            <stringProp name="queryArguments"></stringProp>
            <stringProp name="queryArgumentsTypes"></stringProp>
            <stringProp name="queryTimeout"></stringProp>
            <stringProp name="queryType">Update Statement</stringProp>
            <stringProp name="resultVariable"></stringProp>
            <stringProp name="variableNames"></stringProp>
          </JDBCSampler>
          <hashTree>
            <JSR223PostProcessor guiclass="TestBeanGUI" testclass="JSR223PostProcessor" testname="Set Label to Query Name" enabled="true">
              <stringProp name="cacheKey">true</stringProp>
              <stringProp name="filename"></stringProp>
              <stringProp name="parameters"></stringProp>
              <stringProp name="script">prev.setSampleLabel(vars.get("query_name"));</stringProp>
              <stringProp name="scriptLanguage">groovy</stringProp>
            </JSR223PostProcessor>
            <hashTree/>
          </hashTree>
        <ResultCollector guiclass="SimpleDataWriter" testclass="ResultCollector" testname="Phase 1 Report" enabled="true">
          <boolProp name="ResultCollector.error_logging">false</boolProp>
          <objProp>
            <name>saveConfig</name>
            <value class="SampleSaveConfiguration">
              <time>true</time>
              <latency>true</latency>
              <timestamp>true</timestamp>
              <success>true</success>
              <label>true</label>
              <code>true</code>
              <message>true</message>
              <threadName>true</threadName>
              <dataType>true</dataType>
              <encoding>false</encoding>
              <assertions>true</assertions>
              <subresults>true</subresults>
              <responseData>false</responseData>
              <samplerData>false</samplerData>
              <xml>false</xml>
              <fieldNames>true</fieldNames>
              <responseHeaders>false</responseHeaders>
              <requestHeaders>false</requestHeaders>
              <responseDataOnError>false</responseDataOnError>
              <saveAssertionResultsFailureMessage>true</saveAssertionResultsFailureMessage>
              <assertionsResultsToSave>0</assertionsResultsToSave>
              <bytes>false</bytes>
              <sentBytes>false</sentBytes>
              <url>false</url>
              <threadCounts>false</threadCounts>
              <idleTime>false</idleTime>
              <connectTime>false</connectTime>
            </value>
          </objProp>
          <stringProp name="filename">{CURPATH}/results/{PLATFORM}-ingestion-report.csv</stringProp>
        </ResultCollector>
        <hashTree/>
      </hashTree>
    </hashTree>
  </hashTree>
</jmeterTestPlan>
"""

In [ ]:
_POWER_TEST_TEMPLATE = """
<?xml version="1.0" encoding="UTF-8"?>
<jmeterTestPlan version="1.2" properties="5.0" jmeter="5.6.3">
  <hashTree>
    <TestPlan guiclass="TestPlanGui" testclass="TestPlan" testname="TPC-DS Power Test Plan" enabled="true">
      <stringProp name="TestPlan.comments"></stringProp>
      <boolProp name="TestPlan.functional_mode">false</boolProp>
      <boolProp name="TestPlan.tearDown_on_shutdown">true</boolProp>
      <boolProp name="TestPlan.serialize_threadgroups">true</boolProp>
      <elementProp name="TestPlan.user_defined_variables" elementType="Arguments" guiclass="ArgumentsPanel" testclass="Arguments" testname="User Defined Variables" enabled="true">
        <collectionProp name="Arguments.arguments"/>
      </elementProp>
      <stringProp name="TestPlan.user_define_classpath">{JDBC_DRIVER}</stringProp>
    </TestPlan>
    <hashTree>
      <JDBCDataSource guiclass="TestBeanGUI" testclass="JDBCDataSource" testname="{CONNECTION_LABEL}" enabled="true">
        <stringProp name="dataSource">{DATASOURCE_NAME}</stringProp>
        <stringProp name="poolMax">1</stringProp>
        <stringProp name="timeout">600000</stringProp>
        <stringProp name="trimInterval">60000</stringProp>
        <boolProp name="autocommit">true</boolProp>
        <stringProp name="transactionIsolation">DEFAULT</stringProp>
        <boolProp name="preinit">true</boolProp>
        <stringProp name="initQuery">{INIT_QUERY}</stringProp>
        <boolProp name="keepAlive">true</boolProp>
        <stringProp name="connectionAge">50000000</stringProp>
        <stringProp name="checkQuery">select 1</stringProp>
        <stringProp name="dbUrl">{JDBC_URL}</stringProp>
        <stringProp name="driver">{JDBC_DRIVER_CLASS}</stringProp>
{JDBC_AUTH_BLOCK}
        <stringProp name="TestPlan.comments"></stringProp>
      </JDBCDataSource>
      <hashTree/>
      <!-- ==================================================================================== -->
      <!-- ==                 PHASE 3: SELECT QUERIES (Power Test)                           == -->
      <!-- ==================================================================================== -->
      <ThreadGroup guiclass="ThreadGroupGui" testclass="ThreadGroup" testname="Phase 3 - Power Test" enabled="true">
        <stringProp name="ThreadGroup.on_sample_error">continue</stringProp>
        <elementProp name="ThreadGroup.main_controller" elementType="LoopController" guiclass="LoopControlPanel" testclass="LoopController" testname="Loop Controller" enabled="true">
          <stringProp name="LoopController.loops">-1</stringProp>
          <boolProp name="LoopController.continue_forever">true</boolProp>
        </elementProp>
        <stringProp name="ThreadGroup.num_threads">1</stringProp>
        <stringProp name="ThreadGroup.ramp_time">1</stringProp>
        <boolProp name="ThreadGroup.delayedStart">false</boolProp>
        <boolProp name="ThreadGroup.scheduler">false</boolProp>
        <stringProp name="ThreadGroup.duration"></stringProp>
        <stringProp name="ThreadGroup.delay"></stringProp>
        <boolProp name="ThreadGroup.same_user_on_next_iteration">false</boolProp>
      </ThreadGroup>
      <hashTree>
        <CSVDataSet guiclass="TestBeanGUI" testclass="CSVDataSet" testname="CSV Data Set Config - Select Queries" enabled="true">
          <stringProp name="delimiter">\t</stringProp>
          <stringProp name="fileEncoding">UTF-8</stringProp>
          <stringProp name="filename">{CURPATH}/test_conf/tpcds-queries-perf-{PLATFORM}.tsv</stringProp>
          <boolProp name="ignoreFirstLine">true</boolProp>
          <boolProp name="quotedData">false</boolProp>
          <boolProp name="recycle">false</boolProp>
          <stringProp name="shareMode">shareMode.thread</stringProp>
          <boolProp name="stopThread">true</boolProp>
          <stringProp name="variableNames">query_name,query</stringProp>
        </CSVDataSet>
        <hashTree/>
          <JDBCSampler guiclass="TestBeanGUI" testclass="JDBCSampler" testname="JDBC Request - Select" enabled="true">
            <stringProp name="dataSource">{DATASOURCE_NAME}</stringProp>
            <stringProp name="query">/* ${{query_name}} */
${{query}}</stringProp>
            <stringProp name="queryArguments"></stringProp>
            <stringProp name="queryArgumentsTypes"></stringProp>
            <stringProp name="queryTimeout"></stringProp>
            <stringProp name="queryType">Select Statement</stringProp>
            <stringProp name="resultVariable"></stringProp>
            <stringProp name="variableNames"></stringProp>
          </JDBCSampler>
          <hashTree>
            <JSR223PostProcessor guiclass="TestBeanGUI" testclass="JSR223PostProcessor" testname="Set Label to Query Name" enabled="true">
              <stringProp name="cacheKey">true</stringProp>
              <stringProp name="filename"></stringProp>
              <stringProp name="parameters"></stringProp>
              <stringProp name="script">prev.setSampleLabel(vars.get("query_name"));</stringProp>
              <stringProp name="scriptLanguage">groovy</stringProp>
            </JSR223PostProcessor>
            <hashTree/>
          </hashTree>
        <ResultCollector guiclass="SimpleDataWriter" testclass="ResultCollector" testname="Phase 3 Report" enabled="true">
          <boolProp name="ResultCollector.error_logging">false</boolProp>
          <objProp>
            <name>saveConfig</name>
            <value class="SampleSaveConfiguration">
              <time>true</time>
              <latency>true</latency>
              <timestamp>true</timestamp>
              <success>true</success>
              <label>true</label>
              <code>true</code>
              <message>true</message>
              <threadName>true</threadName>
              <dataType>true</dataType>
              <encoding>false</encoding>
              <assertions>true</assertions>
              <subresults>true</subresults>
              <responseData>false</responseData>
              <samplerData>false</samplerData>
              <xml>false</xml>
              <fieldNames>true</fieldNames>
              <responseHeaders>false</responseHeaders>
              <requestHeaders>false</requestHeaders>
              <responseDataOnError>false</responseDataOnError>
              <saveAssertionResultsFailureMessage>true</saveAssertionResultsFailureMessage>
              <assertionsResultsToSave>0</assertionsResultsToSave>
              <bytes>false</bytes>
              <sentBytes>false</sentBytes>
              <url>false</url>
              <threadCounts>false</threadCounts>
              <idleTime>false</idleTime>
              <connectTime>false</connectTime>
            </value>
          </objProp>
          <stringProp name="filename">{CURPATH}/results/{PLATFORM}-perf-test-report.csv</stringProp>
        </ResultCollector>
        <hashTree/>
      </hashTree>
    </hashTree>
  </hashTree>
</jmeterTestPlan>
"""

In [ ]:
_CONCURRENCY_TEMPLATE = """
<?xml version="1.0" encoding="UTF-8"?>
<jmeterTestPlan version="1.2" properties="5.0" jmeter="5.6.3">
  <hashTree>
    <TestPlan guiclass="TestPlanGui" testclass="TestPlan" testname="TPC-DS Concurrency Test Plan" enabled="true">
      <stringProp name="TestPlan.comments">20 parallel streams against one {PLATFORM} SQL warehouse</stringProp>
      <boolProp name="TestPlan.functional_mode">false</boolProp>
      <boolProp name="TestPlan.tearDown_on_shutdown">true</boolProp>
      <boolProp name="TestPlan.serialize_threadgroups">false</boolProp>
      <elementProp name="TestPlan.user_defined_variables" elementType="Arguments" guiclass="ArgumentsPanel" testclass="Arguments" testname="User Defined Variables" enabled="true">
        <collectionProp name="Arguments.arguments"/>
      </elementProp>
      <stringProp name="TestPlan.user_define_classpath">{JDBC_DRIVER}</stringProp>
    </TestPlan>
    <hashTree>

      <!-- ======================= JDBC CONNECTION CONFIG ======================= -->
      <JDBCDataSource guiclass="TestBeanGUI" testclass="JDBCDataSource" testname="{CONNECTION_LABEL}" enabled="true">
        <stringProp name="dataSource">{DATASOURCE_NAME}</stringProp>
        <stringProp name="poolMax">20</stringProp>
        <stringProp name="timeout">600000</stringProp>
        <stringProp name="trimInterval">60000</stringProp>
        <boolProp name="autocommit">true</boolProp>
        <stringProp name="transactionIsolation">DEFAULT</stringProp>
        <boolProp name="preinit">true</boolProp>
        <stringProp name="initQuery">{INIT_QUERY}</stringProp>
        <boolProp name="keepAlive">true</boolProp>
        <stringProp name="connectionAge">50000000</stringProp>
        <stringProp name="checkQuery">select 1</stringProp>
        <stringProp name="dbUrl">{JDBC_URL}</stringProp>
        <stringProp name="driver">{JDBC_DRIVER_CLASS}</stringProp>
{JDBC_AUTH_BLOCK}
        <stringProp name="TestPlan.comments"></stringProp>
      </JDBCDataSource>
      <hashTree/>

      <!-- ======================= CONCURRENCY: 20 USERS (1 TG) ======================= -->
      <ThreadGroup guiclass="ThreadGroupGui" testclass="ThreadGroup" testname="Concurrency Streams (20 users)" enabled="true">
        <stringProp name="ThreadGroup.on_sample_error">continue</stringProp>
        <elementProp name="ThreadGroup.main_controller" elementType="LoopController" guiclass="LoopControlPanel" testclass="LoopController" testname="Loop Controller" enabled="true">
          <stringProp name="LoopController.loops">-1</stringProp>
          <boolProp name="LoopController.continue_forever">true</boolProp>
        </elementProp>
        <stringProp name="ThreadGroup.num_threads">20</stringProp>
        <stringProp name="ThreadGroup.ramp_time">10</stringProp>
        <boolProp name="ThreadGroup.delayedStart">false</boolProp>
        <boolProp name="ThreadGroup.scheduler">false</boolProp>
        <stringProp name="ThreadGroup.duration"></stringProp>
        <stringProp name="ThreadGroup.delay"></stringProp>
        <boolProp name="ThreadGroup.same_user_on_next_iteration">false</boolProp>
      </ThreadGroup>
      <hashTree>

        <!-- Each thread uses its OWN stream file based on thread number (1..20) -->
        <CSVDataSet guiclass="TestBeanGUI" testclass="CSVDataSet" testname="CSV Data Set Config - Per Thread Stream" enabled="true">
          <stringProp name="delimiter">\t</stringProp>
          <stringProp name="fileEncoding">UTF-8</stringProp>
          <stringProp name="filename">{CURPATH}/tpcds_{PLATFORM}_concurrency_streams/stream_${{__threadNum}}.tsv</stringProp>
          <boolProp name="ignoreFirstLine">true</boolProp>
          <boolProp name="quotedData">false</boolProp>
          <boolProp name="recycle">false</boolProp>
          <stringProp name="shareMode">shareMode.thread</stringProp>
          <boolProp name="stopThread">true</boolProp>
          <stringProp name="variableNames">query_id,query_text,stream_id</stringProp>
        </CSVDataSet>
        <hashTree/>

        <JDBCSampler guiclass="TestBeanGUI" testclass="JDBCSampler" testname="JDBC Request - Concurrency Select" enabled="true">
          <stringProp name="dataSource">{DATASOURCE_NAME}</stringProp>
          <stringProp name="query">/* ${{query_id}} (stream ${{stream_id}}) */
${{query_text}}</stringProp>
          <stringProp name="queryArguments"></stringProp>
          <stringProp name="queryArgumentsTypes"></stringProp>
          <stringProp name="queryTimeout"></stringProp>
          <stringProp name="queryType">Select Statement</stringProp>
          <stringProp name="resultVariable"></stringProp>
          <stringProp name="variableNames"></stringProp>
        </JDBCSampler>
        <hashTree>
          <JSR223PostProcessor guiclass="TestBeanGUI" testclass="JSR223PostProcessor" testname="Set Label to Query ID and Stream" enabled="true">
            <stringProp name="cacheKey">true</stringProp>
            <stringProp name="filename"></stringProp>
            <stringProp name="parameters"></stringProp>
            <stringProp name="script">prev.setSampleLabel(vars.get("query_id") + " (stream " + vars.get("stream_id") + ")");</stringProp>
            <stringProp name="scriptLanguage">groovy</stringProp>
          </JSR223PostProcessor>
          <hashTree/>
        </hashTree>

        <ResultCollector guiclass="SimpleDataWriter" testclass="ResultCollector" testname="Concurrency Report" enabled="true">
          <boolProp name="ResultCollector.error_logging">false</boolProp>
          <objProp>
            <name>saveConfig</name>
            <value class="SampleSaveConfiguration">
              <time>true</time>
              <latency>true</latency>
              <timestamp>true</timestamp>
              <success>true</success>
              <label>true</label>
              <code>true</code>
              <message>true</message>
              <threadName>true</threadName>
              <dataType>true</dataType>
              <encoding>false</encoding>
              <assertions>true</assertions>
              <subresults>true</subresults>
              <responseData>false</responseData>
              <samplerData>false</samplerData>
              <xml>false</xml>
              <fieldNames>true</fieldNames>
              <responseHeaders>false</responseHeaders>
              <requestHeaders>false</requestHeaders>
              <responseDataOnError>false</responseDataOnError>
              <saveAssertionResultsFailureMessage>true</saveAssertionResultsFailureMessage>
              <assertionsResultsToSave>0</assertionsResultsToSave>
              <bytes>false</bytes>
              <sentBytes>false</sentBytes>
              <url>false</url>
              <threadCounts>true</threadCounts>
              <idleTime>false</idleTime>
              <connectTime>true</connectTime>
            </value>
          </objProp>
          <stringProp name="filename">{CURPATH}/results/{PLATFORM}-concurrency-report.csv</stringProp>
        </ResultCollector>
        <hashTree/>

      </hashTree>

    </hashTree>
  </hashTree>
</jmeterTestPlan>
"""

In [ ]:
def run_dbx_optimize_analyze_via_sdk(*, wh_id, host, pat_token,
                                     catalog, schema,
                                     optimize_tsv, analyze_tsv,
                                     poll_interval=5, timeout_seconds=3600,
                                     analyze_concurrency=5):
    """Run OPTIMIZE + ANALYZE statements out-of-band via the Databricks SDK
    Statement Execution API.

    These are *un-metered* setup steps for the benchmark — they exist on the
    Databricks side but Snowflake auto-handles the equivalent (auto-clustering,
    auto-stats), so we don't include them in the timed JMeter workload.

    Execution model:
      * OPTIMIZE statements run SEQUENTIALLY (only the zorder / iceberg_zordered
        strategies emit these; ZORDER layout is written one table at a time).
      * ANALYZE statements run CONCURRENTLY with `analyze_concurrency` streams
        (default 5). ANALYZE is emitted for every strategy and the 24 tables are
        independent, so parallelism cuts wall-clock setup time substantially.

    Reads (query_name, query) pairs from each TSV (header + tab-separated rows),
    submits each statement, polls until terminal, and raises RuntimeError if any
    statement fails.

    Args:
        wh_id: Databricks SQL warehouse ID.
        host:  Workspace URL (https://...).
        pat_token: PAT for auth.
        catalog, schema: Unity Catalog target for the statements.
        optimize_tsv, analyze_tsv: paths to the TSV files written by
            "2 - Databricks - Ingestion Query Preparation".
        poll_interval: seconds between polls of a still-running statement.
        timeout_seconds: per-statement hard timeout.
        analyze_concurrency: number of ANALYZE statements in flight at once.
    """
    import csv as _csv
    import os as _os
    import time
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from databricks.sdk import WorkspaceClient

    def _load_queries(path):
        if not _os.path.exists(path):
            return []
        with open(path) as f:
            reader = _csv.reader(f, delimiter="\t", quoting=_csv.QUOTE_NONE)
            next(reader, None)  # header
            return [(row[0], row[1]) for row in reader if len(row) >= 2 and row[0]]

    optimize_qs = _load_queries(optimize_tsv)
    analyze_qs = _load_queries(analyze_tsv)

    if not optimize_qs and not analyze_qs:
        print("No OPTIMIZE/ANALYZE statements to run (TSVs empty).")
        return

    if host.startswith("jdbc:"):
        host = host.replace("jdbc:databricks:", "https:").split("/sql")[0]
    if not host.startswith("https://"):
        host = "https://" + host

    w = WorkspaceClient(host=host, token=pat_token)

    TERMINAL_OK = {"SUCCEEDED"}
    TERMINAL_BAD = {"FAILED", "CANCELED", "CLOSED"}

    def _run_one(kind, label, sql):
        """Submit one statement and poll to terminal. Returns (label, seconds).
        Raises RuntimeError on failure/timeout."""
        t0 = time.time()
        resp = w.statement_execution.execute_statement(
            statement=sql, warehouse_id=wh_id,
            catalog=catalog, schema=schema,
            wait_timeout="0s",
        )
        sid = resp.statement_id
        deadline = time.time() + timeout_seconds
        while True:
            state = (resp.status.state.value if hasattr(resp.status.state, "value") else str(resp.status.state))
            if state in TERMINAL_OK:
                return label, time.time() - t0
            if state in TERMINAL_BAD:
                err = (resp.status.error.message if (resp.status.error and resp.status.error.message) else state)
                raise RuntimeError(f"[{kind}] {label}: {state} - {err[:300]}")
            if time.time() > deadline:
                raise RuntimeError(f"[{kind}] {label}: timeout after {timeout_seconds}s (last state={state})")
            time.sleep(poll_interval)
            resp = w.statement_execution.get_statement(sid)

    # --- OPTIMIZE: sequential (zorder layout, one table at a time) ---
    if optimize_qs:
        print(f"Running {len(optimize_qs)} OPTIMIZE statements SEQUENTIALLY on warehouse {wh_id}...")
        for label, sql in optimize_qs:
            _, dt = _run_one("optimize", label, sql)
            print(f"  [optimize] {label}: SUCCESS ({dt:.1f}s)")

    # --- ANALYZE: concurrent (independent per-table stats) ---
    if analyze_qs:
        n = len(analyze_qs)
        streams = max(1, int(analyze_concurrency))
        print(f"Running {n} ANALYZE statements CONCURRENTLY ({streams} streams) on warehouse {wh_id}...")
        done = 0
        with ThreadPoolExecutor(max_workers=streams) as pool:
            futures = {pool.submit(_run_one, "analyze", label, sql): label
                       for label, sql in analyze_qs}
            for fut in as_completed(futures):
                label, dt = fut.result()   # re-raises RuntimeError from the worker
                done += 1
                print(f"  [analyze] {label}: SUCCESS ({dt:.1f}s)  [{done}/{n}]")

    total = len(optimize_qs) + len(analyze_qs)
    print(f"\nAll {total} setup statements completed successfully.")
